# R-Build v2.1 — interactive demo

R-Build: fully interactive, modifiable open-source LLM architecture + training-speed-boosting library.
Pipeline: **single cache-pull loop line → parallel token-generating layers bundled → next parallel stage**.

**v2.1 adds full vision-language support** (images & video) plus a ready-made training script (`examples/train_rhododendron_lite.py`).

In [ ]:
!pip -q install git+https://github.com/hanmuyang1-collab/rbuild-engine "ipywidgets>=8.0"

In [ ]:
from rbuild import RBuildConfig, RBuildModel, Trainer, preset, interactive

## 1. Launch the interactive builder (every value modifiable)

In [ ]:
panel = interactive.launch()   # edit model / cache_loop / parallel / memory / vision / train, click "Build model"
cfg = panel.cfg                # the config you edited lives here

## 2. Build from a scale preset and inspect cost (naive vs optimized)

In [ ]:
cfg = preset("s1")            # counter-verified 19.9B total / 3.3B active
print(cfg.report())            # params + naive-vs-optimized cost, always both columns

## 3. Train (toy data) — MoD + fine-grained MoE + Muon + WSD + chunked CE + delta-rule memory

In [ ]:
import torch
cfg = RBuildConfig()           # tiny default config; edit anything
cfg.train.max_steps = 10
cfg.train.grad_accum = 1
model = RBuildModel(cfg)
print({k: f"{v:,}" for k, v in model.num_params().items()})  # counter == actual, verified

V, B, T = cfg.model.vocab_size, 2, 64
batches = [(torch.randint(0, V, (B, T)), torch.randint(0, V, (B, T))) for _ in range(20)]
trainer = Trainer(model, cfg)
hist = trainer.fit(batches)    # logs loss + lr every 10 steps
hist["loss"][-3:]

## 4. Interactive chat (instruct-style, cache-pull loop active)

In [ ]:
# encode: str -> token tensor (1, N); decode: token tensor -> str
# plug in your own tokenizer; demo uses a trivial hash encoder
encode = lambda s: torch.tensor([[abs(hash(s)) % V]], dtype=torch.long)
decode = lambda ids: " ".join(map(str, ids.tolist()))
interactive.chat(model, encode, decode, max_new_tokens=16)
# commands: /remember /temp /topp /topk /maxn /forget /config /quit

## 4b. VL mode (v2.1)
Enable the vision tower, splice image soft-tokens at `<image>` placeholders, train or generate.

In [ ]:
vcfg = RBuildConfig()                 # small demo config; scale up with preset("s1") etc.
vcfg.vision.enabled = True
vcfg.vision.vit_heads = 8             # must divide d_model (256 here)
vcfg.vision.image_token_id = 31000    # reserved placeholder id (< vocab_size)

vmodel = RBuildModel(vcfg)
counts = vcfg.count_parameters()
print({k: f"{v:,}" for k, v in counts.items()})   # counter-verified: vision params included

tpi = (vcfg.vision.image_size // vcfg.vision.patch_size) ** 2   # tokens per image = (224/14)^2 = 256
ids = torch.randint(0, 31000, (1, tpi + 10))
ids[0, 5:5+tpi] = vcfg.vision.image_token_id                    # one placeholder per vision soft token
images = torch.randn(1, 1, 3, vcfg.vision.image_size, vcfg.vision.image_size)  # (B, N, C, H, W); N>1 = video

logits, loss = vmodel(ids, targets=ids, images=images)          # loss auto-masked at vision positions
print("logits:", tuple(logits.shape), "loss:", round(float(loss), 4))

# Blind mode (default): vision.enabled=False -> no tower built, pure text.

## 5. Notes
- `interactive.launch()` falls back to console prompts outside Jupyter.
- Cost report is always two columns: naive baseline vs R-Build optimized.
- Blind (text-only) by default; flip `cfg.vision.enabled = True` for VL. `freeze_vision=True` trains only the projector.